# Healthcare AI - Model Training

This notebook trains an XGBoost Classifier for disease prediction.

## Section 1: Import libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report
import joblib
import warnings
warnings.filterwarnings('ignore')

## Section 2: Load dataset

In [2]:
df = pd.read_csv('data/Testing.csv')
df.head()

,itching,skin_rash,nodal_skin_eruptions,continuous_sneezing,shivering,chills,joint_pain,stomach_pain,acidity,ulcers_on_tongue,...,blackheads,scurring,skin_peeling,silver_like_dusting,small_dents_in_nails,inflammatory_nails,blister,red_sore_around_nose,yellow_crust_ooze,prognosis
0,1,1,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Fungal infection
1,0,0,0,1,1,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Allergy
2,0,0,0,0,0,0,0,1,1,1,...,0,0,0,0,0,0,0,0,0,GERD
3,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Chronic cholestasis
4,1,1,0,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,Drug Reaction


## Section 3: Data understanding

In [3]:
print(f'Dataset Shape: {df.shape}')
print('\nData Types:')
print(df.dtypes)
print('\nSummary Statistics:')
display(df.describe())
print('\nClass Distribution:')
print(df.iloc[:, -1].value_counts())

Dataset Shape: (42, 133)

Data Types:
itching                  int64
skin_rash                int64
nodal_skin_eruptions     int64
continuous_sneezing      int64
shivering                int64
                         ...  
inflammatory_nails       int64
blister                  int64
red_sore_around_nose     int64
yellow_crust_ooze        int64
prognosis               object
Length: 133, dtype: object

Summary Statistics:


,itching,skin_rash,nodal_skin_eruptions,continuous_sneezing,shivering,chills,joint_pain,stomach_pain,acidity,ulcers_on_tongue,...,pus_filled_pimples,blackheads,scurring,skin_peeling,silver_like_dusting,small_dents_in_nails,inflammatory_nails,blister,red_sore_around_nose,yellow_crust_ooze
count,42.000000,42.000000,42.000000,42.000000,42.000000,42.000000,42.000000,42.000000,42.000000,42.000000,...,42.000000,42.000000,42.000000,42.000000,42.000000,42.000000,42.000000,42.000000,42.000000,42.000000
mean,0.166667,0.190476,0.023810,0.047619,0.023810,0.166667,0.142857,0.047619,0.047619,0.023810,...,0.023810,0.023810,0.023810,0.047619,0.023810,0.023810,0.023810,0.023810,0.047619,0.023810
std,0.377195,0.397437,0.154303,0.215540,0.154303,0.377195,0.354169,0.215540,0.215540,0.154303,...,0.154303,0.154303,0.154303,0.215540,0.154303,0.154303,0.154303,0.154303,0.215540,0.154303
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
max,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000



Class Distribution:
prognosis
Fungal infection                           2
Hepatitis C                                1
Hepatitis E                                1
Alcoholic hepatitis                        1
Tuberculosis                               1
Common Cold                                1
Pneumonia                                  1
Dimorphic hemmorhoids(piles)               1
Heart attack                               1
Varicose veins                             1
Hypothyroidism                             1
Hyperthyroidism                            1
Hypoglycemia                               1
Osteoarthristis                            1
Arthritis                                  1
(vertigo) Paroymsal  Positional Vertigo    1
Acne                                       1
Urinary tract infection                    1
Psoriasis                                  1
Hepatitis D                                1
Hepatitis B                                1
Allergy                 

## Section 4: Missing value analysis

In [4]:
missing_values = df.isnull().sum()
print('Missing values per column:')
print(missing_values[missing_values > 0])

if df.isnull().sum().sum() == 0:
    print('No missing values found in the dataset.')
else:
    print(f'Total missing values: {df.isnull().sum().sum()}')

Missing values per column:
Series([], dtype: int64)
No missing values found in the dataset.


## Section 5: Data preprocessing

In [5]:
X = df.drop('prognosis', axis=1)
y = df['prognosis']
features = list(X.columns)
print(f'Training on {len(X)} samples.')

Training on 42 samples.


## Section 6: Feature Encoding

In [6]:
le = LabelEncoder()
y_encoded = le.fit_transform(y)
print(f'Encoded {len(le.classes_)} unique classes.')

Encoded 41 unique classes.


## Section 7: Train/Test Split

In [7]:
print('Using the entire dataset for training since it is small.')

Using the entire dataset for training since it is small.


## Section 8: Train the selected model
Training an XGBoost Classifier

In [8]:
model = MultinomialNB()
model.fit(X, y_encoded)
print('Model training complete with MultinomialNB.')

Model training complete with MultinomialNB.


## Section 9: Evaluate the model

In [9]:
y_pred = model.predict(X)
print('Accuracy on training set: ', accuracy_score(y_encoded, y_pred))

Accuracy on training set:  1.0


## Section 10: Feature Importance

In [10]:
print('MultinomialNB does not have feature_importances_ like tree models.')

MultinomialNB does not have feature_importances_ like tree models.


## Section 11: Save model using joblib

In [11]:
model_data = {
    'model': model,
    'label_encoder': le,
    'features': features
}
joblib.dump(model_data, 'trained_model.pkl')
print('Model saved successfully as trained_model.pkl')

Model saved successfully as trained_model.pkl
